In [28]:
!pip install torch torch-geometric

In [29]:
import os
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from torch_geometric.data import Data

In [30]:
NODES_PATH = "../data/processed/final_nodes.csv"
EDGES_PATH = "../data/processed/graph_edges.csv"

OUTPUT_PATH = "../data/processed/graph_data.pt"

In [31]:
nodes_df = pd.read_csv(NODES_PATH)
edges_df = pd.read_csv(EDGES_PATH)

print("Nodes shape:", nodes_df.shape)
print("Edges shape:", edges_df.shape)

display(nodes_df.head())
display(edges_df.head())

Nodes shape: (4404, 139)
Edges shape: (50000, 9)


,node_id,ip_address,source_Flow Duration,source_Total Fwd Packets,source_Total Backward Packets,source_Total Length of Fwd Packets,source_Total Length of Bwd Packets,source_Fwd Packet Length Max,source_Fwd Packet Length Min,source_Fwd Packet Length Mean,...,destination_Active Std,destination_Active Max,destination_Active Min,destination_Idle Mean,destination_Idle Std,destination_Idle Max,destination_Idle Min,source_flow_count,destination_flow_count,node_label
0,0,101.37.178.168,-0.263543,-0.032299,-0.110270,-0.067572,-0.422974,-0.306618,-0.019509,-0.190242,...,0.369613,0.164335,-0.035071,-0.319954,0.006259,-0.314945,-0.350110,-0.026136,-0.019858,0
1,1,103.203.138.103,-0.358915,-0.032299,-0.844265,-0.067572,-0.428341,-0.306618,-0.019509,-0.190242,...,-0.187482,0.012134,0.068937,-0.470509,-0.202108,-0.474897,-0.437191,-0.024911,-0.024230,0
2,2,103.23.4.38,-0.358938,-0.032299,-0.844265,-0.054079,-0.428341,-0.059807,-0.231231,-0.027564,...,-0.416458,-0.298673,-0.112401,-0.879101,-0.204858,-0.869199,-0.852831,-0.026136,-0.025025,0
3,3,103.235.247.8,-0.019542,-0.276015,-0.110270,-0.033343,-0.368414,0.216622,0.029892,0.275531,...,0.867312,0.302114,-0.056125,-0.610382,-0.196585,-0.609392,-0.581360,-0.024502,-0.024230,0
4,4,103.235.252.17,-0.358913,-0.438492,-0.110270,-0.076094,-0.428341,-0.365852,-0.231231,-0.292986,...,-0.139769,-0.105266,-0.062587,-0.529074,-0.049305,-0.522790,-0.552746,-0.026136,-0.024628,0


,source,target,source_ip,destination_ip,source_port,destination_port,protocol,timestamp,label
0,990,994,192.168.10.17,192.168.10.3,50251,53,17,7/7/2017 1:48,0
1,639,996,172.16.0.1,192.168.10.50,42254,32777,6,7/7/2017 3:08,1
2,639,996,172.16.0.1,192.168.10.50,52754,30951,6,7/7/2017 2:55,1
3,639,996,172.16.0.1,192.168.10.50,43109,902,6,7/7/2017 2:51,1
4,639,996,172.16.0.1,192.168.10.50,37796,500,6,7/7/2017 2:52,1


In [32]:
print("Node columns:")
print(nodes_df.columns.tolist())

print("\nEdge columns:")
print(edges_df.columns.tolist())

Node columns:
['node_id', 'ip_address', 'source_Flow Duration', 'source_Total Fwd Packets', 'source_Total Backward Packets', 'source_Total Length of Fwd Packets', 'source_Total Length of Bwd Packets', 'source_Fwd Packet Length Max', 'source_Fwd Packet Length Min', 'source_Fwd Packet Length Mean', 'source_Fwd Packet Length Std', 'source_Bwd Packet Length Max', 'source_Bwd Packet Length Min', 'source_Bwd Packet Length Mean', 'source_Bwd Packet Length Std', 'source_Flow Bytes/s', 'source_Flow Packets/s', 'source_Flow IAT Mean', 'source_Flow IAT Std', 'source_Flow IAT Max', 'source_Flow IAT Min', 'source_Fwd IAT Total', 'source_Fwd IAT Mean', 'source_Fwd IAT Std', 'source_Fwd IAT Max', 'source_Fwd IAT Min', 'source_Bwd IAT Total', 'source_Bwd IAT Mean', 'source_Bwd IAT Std', 'source_Bwd IAT Max', 'source_Bwd IAT Min', 'source_Fwd PSH Flags', 'source_Fwd Header Length', 'source_Bwd Header Length', 'source_Fwd Packets/s', 'source_Bwd Packets/s', 'source_Min Packet Length', 'source_Max Packet

In [33]:
source_column = "source"
destination_column = "target"
label_column = "node_label"

print("Source column:", source_column)
print("Destination column:", destination_column)
print("Label column:", label_column)

Source column: source
Destination column: target
Label column: node_label


In [34]:
required_node_columns = [
    "node_id",
    "ip_address",
    "node_label"
]

required_edge_columns = [
    "source",
    "target"
]

for column in required_node_columns:
    if column not in nodes_df.columns:
        raise ValueError(f"Missing node column: {column}")

for column in required_edge_columns:
    if column not in edges_df.columns:
        raise ValueError(f"Missing edge column: {column}")

print("All required columns exist.")

All required columns exist.


In [35]:
number_of_nodes = len(nodes_df)

print("Number of nodes:", number_of_nodes)

print("Minimum source ID:", edges_df[source_column].min())
print("Maximum source ID:", edges_df[source_column].max())

print("Minimum target ID:", edges_df[destination_column].min())
print("Maximum target ID:", edges_df[destination_column].max())

Number of nodes: 4404
Minimum source ID: 1
Maximum source ID: 2998
Minimum target ID: 0
Maximum target ID: 3000


In [36]:
excluded_columns = [
    "node_id",
    "ip_address",
    "node_label"
]

feature_columns = [
    column
    for column in nodes_df.columns
    if column not in excluded_columns
]

print("Number of node features:", len(feature_columns))
print(feature_columns)

Number of node features: 136
['source_Flow Duration', 'source_Total Fwd Packets', 'source_Total Backward Packets', 'source_Total Length of Fwd Packets', 'source_Total Length of Bwd Packets', 'source_Fwd Packet Length Max', 'source_Fwd Packet Length Min', 'source_Fwd Packet Length Mean', 'source_Fwd Packet Length Std', 'source_Bwd Packet Length Max', 'source_Bwd Packet Length Min', 'source_Bwd Packet Length Mean', 'source_Bwd Packet Length Std', 'source_Flow Bytes/s', 'source_Flow Packets/s', 'source_Flow IAT Mean', 'source_Flow IAT Std', 'source_Flow IAT Max', 'source_Flow IAT Min', 'source_Fwd IAT Total', 'source_Fwd IAT Mean', 'source_Fwd IAT Std', 'source_Fwd IAT Max', 'source_Fwd IAT Min', 'source_Bwd IAT Total', 'source_Bwd IAT Mean', 'source_Bwd IAT Std', 'source_Bwd IAT Max', 'source_Bwd IAT Min', 'source_Fwd PSH Flags', 'source_Fwd Header Length', 'source_Bwd Header Length', 'source_Fwd Packets/s', 'source_Bwd Packets/s', 'source_Min Packet Length', 'source_Max Packet Length', 

In [37]:
X = nodes_df[feature_columns].copy()

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)
X = X.astype(np.float32)

x = torch.tensor(
    X.values,
    dtype=torch.float32
)

y = torch.tensor(
    nodes_df[label_column].values,
    dtype=torch.long
)

print("Node feature tensor shape:", x.shape)
print("Node label tensor shape:", y.shape)

Node feature tensor shape: torch.Size([4404, 136])
Node label tensor shape: torch.Size([4404])


In [38]:
edge_source = edges_df[source_column].astype(int).values
edge_destination = edges_df[destination_column].astype(int).values

edge_index = torch.tensor(
    [
        edge_source,
        edge_destination
    ],
    dtype=torch.long
)

print("Edge index shape:", edge_index.shape)

Edge index shape: torch.Size([2, 50000])


In [39]:
invalid_source_nodes = (
    (edge_index[0] < 0) |
    (edge_index[0] >= number_of_nodes)
)

invalid_destination_nodes = (
    (edge_index[1] < 0) |
    (edge_index[1] >= number_of_nodes)
)

if invalid_source_nodes.any():
    raise ValueError("Invalid source node IDs found.")

if invalid_destination_nodes.any():
    raise ValueError("Invalid destination node IDs found.")

print("All edge node IDs are valid.")

All edge node IDs are valid.


In [41]:
# Cell 13 — Create train, validation, and test masks

from sklearn.model_selection import train_test_split
import numpy as np
import torch

node_labels = y.numpy()
node_indices = np.arange(len(node_labels))

print("Total nodes:", len(node_labels))
print("Class distribution:")
print(pd.Series(node_labels).value_counts())

# Check whether stratified splitting is possible
class_counts = pd.Series(node_labels).value_counts()

if class_counts.min() >= 3:

    # 70% training, 30% temporary
    train_indices, temporary_indices = train_test_split(
        node_indices,
        test_size=0.30,
        random_state=42,
        stratify=node_labels
    )

    # 15% validation, 15% testing
    validation_indices, test_indices = train_test_split(
        temporary_indices,
        test_size=0.50,
        random_state=42,
        stratify=node_labels[temporary_indices]
    )

else:

    print("Warning: Stratified splitting is not possible.")
    print("Using random splitting instead.")

    train_indices, temporary_indices = train_test_split(
        node_indices,
        test_size=0.30,
        random_state=42
    )

    validation_indices, test_indices = train_test_split(
        temporary_indices,
        test_size=0.50,
        random_state=42
    )

# Create boolean masks
train_mask = torch.zeros(len(node_labels), dtype=torch.bool)
val_mask = torch.zeros(len(node_labels), dtype=torch.bool)
test_mask = torch.zeros(len(node_labels), dtype=torch.bool)

train_mask[train_indices] = True
val_mask[validation_indices] = True
test_mask[test_indices] = True

print("Training nodes:", train_mask.sum().item())
print("Validation nodes:", val_mask.sum().item())
print("Testing nodes:", test_mask.sum().item())

print("\nTraining labels:")
print(y[train_mask].bincount(minlength=2))

print("\nValidation labels:")
print(y[val_mask].bincount(minlength=2))

print("\nTesting labels:")
print(y[test_mask].bincount(minlength=2))

Total nodes: 4404
Class distribution:
0    4402
1       2
Name: count, dtype: int64
Using random splitting instead.
Training nodes: 3082
Validation nodes: 661
Testing nodes: 661

Training labels:
tensor([3081,    1])

Validation labels:
tensor([660,   1])

Testing labels:
tensor([661,   0])


In [43]:
# Cell 14 — Create the graph object

graph_data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

print(graph_data)

Data(x=[4404, 136], edge_index=[2, 50000], y=[4404], train_mask=[4404], val_mask=[4404], test_mask=[4404])


In [ ]:
# Cell 15 — Verify graph structure

print("Number of nodes:", graph_data.num_nodes)
print("Number of edges:", graph_data.num_edges)
print("Number of node features:", graph_data.num_node_features)

print("Training nodes:", graph_data.train_mask.sum().item())
print("Validation nodes:", graph_data.val_mask.sum().item())
print("Testing nodes:", graph_data.test_mask.sum().item())

print("Graph is valid:", graph_data.validate())

In [ ]:
print("All nodes:")
print(torch.bincount(graph_data.y))

print("\nTraining nodes:")
print(torch.bincount(graph_data.y[graph_data.train_mask]))

print("\nValidation nodes:")
print(torch.bincount(graph_data.y[graph_data.val_mask]))

print("\nTesting nodes:")
print(torch.bincount(graph_data.y[graph_data.test_mask]))

In [44]:
# Cell 16 — Check class distribution

print("All labels:")
print(torch.bincount(graph_data.y, minlength=2))

print("\nTraining labels:")
print(torch.bincount(graph_data.y[graph_data.train_mask], minlength=2))

print("\nValidation labels:")
print(torch.bincount(graph_data.y[graph_data.val_mask], minlength=2))

print("\nTesting labels:")
print(torch.bincount(graph_data.y[graph_data.test_mask], minlength=2))

All labels:
tensor([4402,    2])

Training labels:
tensor([3081,    1])

Validation labels:
tensor([660,   1])

Testing labels:
tensor([661,   0])


In [45]:
# Cell 17 — Save the graph

OUTPUT_PATH = "../data/processed/graph_data.pt"

torch.save(graph_data, OUTPUT_PATH)

print("Graph saved successfully:")
print(OUTPUT_PATH)

Graph saved successfully:
../data/processed/graph_data.pt


In [46]:
# Cell 18 — Reload and verify saved graph

loaded_graph = torch.load(
    OUTPUT_PATH,
    weights_only=False
)

print(loaded_graph)
print("Reload successful.")
print("Nodes:", loaded_graph.num_nodes)
print("Edges:", loaded_graph.num_edges)
print("Features:", loaded_graph.num_node_features)

Data(x=[4404, 136], edge_index=[2, 50000], y=[4404], train_mask=[4404], val_mask=[4404], test_mask=[4404])
Reload successful.
Nodes: 4404
Edges: 50000
Features: 136


In [47]:
# Cell 19 — Final verification

assert graph_data.x.shape[0] == graph_data.y.shape[0]
assert graph_data.edge_index.shape[0] == 2
assert graph_data.train_mask.shape[0] == graph_data.num_nodes
assert graph_data.val_mask.shape[0] == graph_data.num_nodes
assert graph_data.test_mask.shape[0] == graph_data.num_nodes

print("DAY 5 COMPLETED SUCCESSFULLY")
print("Graph is ready for GNN training.")

DAY 5 COMPLETED SUCCESSFULLY
Graph is ready for GNN training.
